# Mathematical Engineering - Financial Engineering, FY 2025-2026

# Risk Management - Assignment 1: Hedging a Swaption Portfolio

**Case study:** The IR-derivative desk of Polimi Bank has the following positions opened today (15/02/2008):

- Long swaption payer 1m&10y 5y ATM - Notional €650 Mln
- Vanilla 10y IRS fixed rate receiver - Notional €550 Mln


In [82]:
# Importing the libraries
import numpy as np
import pandas as pd

import datetime as dt

from utilities.date_functions import (
    business_date_offset,
    date_series,
    year_frac_act_x,
)
from utilities.ex0_utilities import bootstrap
from utilities.ex1_utilities import (
    swaption_price_calculator,
    irs_proxy_duration,
    swap_par_rate,
    swap_mtm,
    SwapType,
)

In [83]:
# =============================================================================
# Portfolio Parameters
# =============================================================================
       # volatilité réaliste
# Swaption
swaption_maturity_y = 10  # Years component of swaption expiry
swaption_maturity_m = 1  # Months component of swaption expiry
swaption_tenor_y = 5  # Underlying swap tenor in years
swaption_fixed_leg_freq = 1  # Annual fixed leg payments
swaption_type = SwapType.RECEIVER
swaption_notional = 650_000_000
sigma_black = 0.7955  # Black implied volatility for the swaption

# IRS
irs_maturity = 10  # In years
irs_fixed_leg_freq = 1  # Annual fixed leg payments
irs_notional = 550_000_000
irs_type = SwapType.RECEIVER

In [84]:
# =============================================================================
# Bootstrap the discount curve from market data (Assignment 0)
# =============================================================================

# !!! COMPLETE AS APPROPRIATE !!!

today = dt.date(2008,2,15)
settlement = dt.date(2008,2,19)

dt = pd.read_csv('mkt_data/dt.csv',
                index_col = 'Market',
                usecols = ['Market','TARGET'],
                converters = {'TARGET':pd.to_datetime})
settlement_date  = dt.TARGET['Settlement']

depo_converter = lambda x: float(x)/100.0  # apply the correct one

df_depos = pd.read_csv('mkt_data/depos.csv', 
                   index_col ='Depos',
                   usecols = ['Depos','ASK','BID'], 
                   converters={'Depos':pd.to_datetime,'BID':depo_converter,'ASK':depo_converter})

future_converter = lambda x: float(x) # apply the correct one

futures = pd.read_csv('mkt_data/futures.csv',
                      index_col ='Futures',
                      usecols = ['Futures','ASK','BID'],
                      converters={'Futures':pd.to_datetime,'BID':future_converter,'ASK':future_converter})
expiry = pd.read_csv('mkt_data/expiry.csv',
                     index_col = 'Futures',
                     usecols =['Futures', 'Settle', 'Expiry'], 
                     converters = {'Futures':pd.to_datetime, 'Settle':pd.to_datetime, 'Expiry':pd.to_datetime})
df_futures = futures.join(expiry)

swap_converter = lambda x: float(x) # apply the correct one

df_swaps = pd.read_csv('mkt_data/swaps.csv',
                    index_col = 'Swaps',
                    usecols = ['Swaps','BID','ASK'],
                    converters={'Swaps':pd.to_datetime,'BID':swap_converter,'ASK':swap_converter})


settlement_date = dt.TARGET['Settlement']

discount_factors, zero_rates = bootstrap(settlement_date, df_depos, df_futures, df_swaps)

## Q1: Mark-to-Market the portfolio at the mid-rate curve


In [85]:
# =============================================================================
# Q1: Compute the forward swap rate (= swaption strike, since ATM)
# =============================================================================

# Swaption expiry: today + 10y1m
swaption_expiry = business_date_offset(
    today, year_offset=swaption_maturity_y, month_offset=swaption_maturity_m
)

# Underlying swap expiry: swaption expiry + 5y tenor
underlying_expiry = business_date_offset(
    today,
    year_offset=swaption_maturity_y + swaption_tenor_y,
    month_offset=swaption_maturity_m,
)

# Fixed leg schedule of the underlying forward-starting swap, in business dates
swaption_underlying_fixed_leg_schedule = date_series(
    swaption_expiry, underlying_expiry, swaption_fixed_leg_freq
)
# Forward swap rate
fwd_swap_rate = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors,
    swaption_underlying_fixed_leg_schedule[0],
)
print(f"Forward swap rate: {fwd_swap_rate:.3%}")

Forward swap rate: 5.300%


In [86]:
# =============================================================================
# Q1: Portfolio MtM
# =============================================================================

strike = fwd_swap_rate # !!! COMPLETE AS APPROPRIATE !!!
swaption_type = SwapType.PAYER 

swaption_price, swaption_delta = swaption_price_calculator(
    fwd_swap_rate,
    strike,
    today,
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors,
    swaption_type,
    compute_delta=True,
)

irs_expiry = business_date_offset(today, year_offset=irs_maturity)
irs_fixed_leg_payment_dates = date_series(today, irs_expiry, irs_fixed_leg_freq)[1:]
irs_rate = swap_par_rate (irs_fixed_leg_payment_dates, discount_factors)
print(f"IRS par rate: {irs_rate:.3%}")
irs_mtm = swap_mtm(irs_rate, irs_fixed_leg_payment_dates, discount_factors, SwapType.RECEIVER)

# Portfolio MtM = swaption value + IRS value
ptf_mtm = swaption_notional * swaption_price + irs_notional * irs_mtm
print(f"Swaption price (per unit notional): {swaption_price:.6f}")
print(f"IRS MtM (per unit notional):        {irs_mtm:.6f}")
print(f"Portfolio MtM:                       \u20ac{ptf_mtm:,.2f}")

IRS par rate: 4.433%
Swaption price (per unit notional): 0.115962
IRS MtM (per unit notional):        0.000000
Portfolio MtM:                       €75,375,398.94


## Q2: Evaluate the portfolio DV01-parallel


In [87]:
# =============================================================================
# Q2: Portfolio DV01-parallel (numerical)
# =============================================================================
df_depos_up = df_depos.copy()
df_depos_up['BID'] += 0.01
df_depos_up['ASK'] += 0.01
df_futures_up = df_futures.copy()
# for the futures we have to subtract the shock, since the price is quoted as 100 - rate   
df_futures_up['BID'] -= 0.01
df_futures_up['ASK'] -= 0.01
df_swaps_up = df_swaps.copy()
df_swaps_up['BID'] += 0.01
df_swaps_up['ASK'] += 0.01

discount_factors_up, zero_rates_up = bootstrap(settlement_date, df_depos_up, df_futures_up, df_swaps_up)

fwd_swap_rate_up = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_up,
    swaption_underlying_fixed_leg_schedule[0],
)
print(f"Forward swap rate under the shocked curve: {fwd_swap_rate_up:.3%}")

# Swaption price under the shocked curve
swaption_price_up = swaption_price_calculator(
    fwd_swap_rate_up,
    strike,
    today,
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors_up,
    swaption_type,
)
print(f"Swaption price under the shocked curve (per unit notional): {swaption_price_up:.6f}")

irs_mtm_up = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_up, swap_type=SwapType.RECEIVER
)
print(f"IRS MtM under the shocked curve (per unit notional): {irs_mtm_up:.6f}")
# Shocked portfolio MtM
ptf_mtm_up = swaption_notional * swaption_price_up + irs_notional * irs_mtm_up
print(f"Portfolio MtM under the shocked curve: \u20ac{ptf_mtm_up:,.2f}")
# DV01-parallel
ptf_numeric_dv01 = ptf_mtm_up - ptf_mtm
print(f"Portfolio DV01-parallel: \u20ac{ptf_numeric_dv01:,.2f}")

Forward swap rate under the shocked curve: 5.311%
Swaption price under the shocked curve (per unit notional): 0.116078
IRS MtM under the shocked curve (per unit notional): -0.000803
Portfolio MtM under the shocked curve: €75,008,684.30
Portfolio DV01-parallel: €-366,714.63


## Q3: Analytical portfolio DV01 approximation


In [88]:
# =============================================================================
# Q3: Analytical portfolio DV01
# =============================================================================

irs_duration = irs_proxy_duration(
    today, irs_rate, irs_fixed_leg_payment_dates, discount_factors
)

ptf_proxy_dv01 = (
    swaption_notional * swaption_delta - irs_notional * irs_duration
) * 1e-4

print(f"Swaption delta: {swaption_delta:.6f}")
print(f"IRS duration:   {irs_duration:.6f}")

print(f"Portfolio proxy DV01:    \u20ac{ptf_proxy_dv01:,.2f}")
print(f"Portfolio numeric DV01:  \u20ac{ptf_numeric_dv01:,.2f}")
print(f"Difference:             \u20ac{ptf_proxy_dv01 - ptf_numeric_dv01:,.2f}")

Swaption delta: 2.472640
IRS duration:   8.270339
Portfolio proxy DV01:    €-294,147.06
Portfolio numeric DV01:  €-366,714.63
Difference:             €72,567.58


## Q4: Delta-hedge the portfolio with a 10y IRS


In [89]:
# =============================================================================
# Q4: Delta hedging with a 10y IRS
# =============================================================================

min_lot = 1_000_000  # IRS traded in multiples of €1M 

# Round to the nearest €1M lot (=> there will be a small residual)
delta_hedge_swap_notional = -round(ptf_numeric_dv01 / (irs_mtm_up - irs_mtm) / min_lot) * min_lot

# IRS receiver (already in portfolio) under shocked curve
irs_mtm_receiver_up = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_up, swap_type=SwapType.RECEIVER
)

# IRS payer (hedge) under shocked curve
irs_mtm_payer_up = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_up, swap_type=SwapType.PAYER
)

# Verify: hedged portfolio DV01 should be ~0
delta_hedge_dv01 = (
    swaption_notional * swaption_price_up          # swaption under shocked curve
    + irs_notional * irs_mtm_receiver_up           # IRS receiver (already in portfolio)
    + delta_hedge_swap_notional * irs_mtm_payer_up # IRS hedge (payer)
) - ptf_mtm
print(
    f"Numerical hedge: \u20ac{delta_hedge_swap_notional:,.0f} total IRS notional, residual DV01: \u20ac{delta_hedge_dv01:,.0f}"
)

# --- Analytical hedge (using the proxy DV01) ---

# DV01 per unit notional of the hedge IRS (payer): opposite sign to receiver
irs_dv01_per_unit_proxy = -irs_duration * 1e-4

delta_hedge_swap_notional_proxy = round(-ptf_proxy_dv01 / irs_dv01_per_unit_proxy / min_lot) * min_lot

print(
    f"Analytical hedge: €{delta_hedge_swap_notional_proxy:,.0f} total IRS notional"
)
print(
    "\nThe analytical approximation significantly overestimates the required hedge notional."
)

Numerical hedge: €-457,000,000 total IRS notional, residual DV01: €-733,748
Analytical hedge: €-356,000,000 total IRS notional

The analytical approximation significantly overestimates the required hedge notional.


## Q5: Portfolio coarse-grained bucket DV01 (10y and 15y buckets)


In [90]:
# =============================================================================
# Q5: Coarse-Grained Bucket DV01 construction
# =============================================================================

def piecewise_linear_weights(dates, reference_date, key_rates):
    year_fracs = np.array([year_frac_act_x(reference_date, d, 365) for d in dates])
    
    key_rates = np.array(sorted(key_rates))
    n = len(year_fracs)
    m = len(key_rates)
    
    weights = np.zeros((m, n))
    
    for i, yf in enumerate(year_fracs):
        # avant le premier point → tout au premier bucket
        if yf <= key_rates[0]:
            weights[0, i] = 1
        
        # après le dernier point → tout au dernier bucket
        elif yf >= key_rates[-1]:
            weights[-1, i] = 1
        
        else:
            # interpolation entre deux key rates
            for j in range(m - 1):
                if key_rates[j] <= yf <= key_rates[j+1]:
                    w_right = (yf - key_rates[j]) / (key_rates[j+1] - key_rates[j])
                    w_left = 1 - w_right
                    
                    weights[j, i] = w_left
                    weights[j+1, i] = w_right
                    break
    
    return weights

key_rates = [10, 15]

weights = piecewise_linear_weights(df_swaps.index, settlement_date, key_rates)

bump = 0.01

shock_swaps_10y = pd.Series(
    bump * weights[0],
    index=df_swaps.index
)

df_swaps_10y_up = df_swaps.copy()
df_swaps_10y_up['BID'] += shock_swaps_10y
df_swaps_10y_up['ASK'] += shock_swaps_10y

discount_factors_10y_up, _ = bootstrap(
    settlement_date, df_depos, df_futures, df_swaps_10y_up
)

shock_swaps_15y = pd.Series(
    bump * weights[1],
    index=df_swaps.index
)

df_swaps_15y_up = df_swaps.copy()
df_swaps_15y_up['BID'] += shock_swaps_15y
df_swaps_15y_up['ASK'] += shock_swaps_15y

discount_factors_15y_up, _ = bootstrap(
    settlement_date, df_depos, df_futures, df_swaps_15y_up
)




In [91]:
# =============================================================================
# Q5: Portfolio Coarse-Grained 10y Bucket DV01
# =============================================================================

fwd_swap_rate_10y_up=swap_par_rate(swaption_underlying_fixed_leg_schedule[1:], discount_factors_10y_up, swaption_underlying_fixed_leg_schedule[0]) #new function


swaption_price_10y_up = swaption_price_calculator(
    fwd_swap_rate_10y_up, #modificata
    strike,
    today, #settlment day
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors_10y_up,
    swaption_type,
)


# IRS MtM under 10y bucket shock
irs_mtm_10y_up = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_10y_up, swap_type=irs_type
)

ptf_mtm_10y_up = (
    swaption_notional * swaption_price_10y_up + irs_notional * irs_mtm_10y_up
)

ptf_numeric_10y_dv01 = ptf_mtm_10y_up - ptf_mtm
print(f"Portfolio Coarse-Grained 10y Bucket DV01: \u20ac{ptf_numeric_10y_dv01:,.2f}")

Portfolio Coarse-Grained 10y Bucket DV01: €-923,907.04


In [92]:
# Diagnostic des signes individuels
print(f"Swaption base:    {swaption_notional * swaption_price:,.2f}")
print(f"IRS base:         {irs_notional * irs_mtm:,.2f}")

# Après shock
sw_dv01  = swaption_notional * (swaption_price_10y_up - swaption_price)
irs_dv01 = irs_notional      * (irs_mtm_10y_up        - irs_mtm)

print(f"Swaption DV01 10y: {sw_dv01:,.2f}")   # attendu: positif
print(f"IRS DV01 10y:      {irs_dv01:,.2f}")  # attendu: négatif
print(f"Portfolio DV01 10y:{sw_dv01 + irs_dv01:,.2f}")


Swaption base:    75,375,398.94
IRS base:         0.00
Swaption DV01 10y: -484,247.73
IRS DV01 10y:      -439,659.31
Portfolio DV01 10y:-923,907.04


In [93]:
print(f"fwd_rate base:     {fwd_swap_rate:.6f}")
print(f"fwd_rate 10y up:   {fwd_swap_rate_10y_up:.6f}")   # doit être > base
print(f"swaption base:     {swaption_price:.8f}")
print(f"swaption 10y up:   {swaption_price_10y_up:.8f}")  # doit être > base
print(f"S0 vs K:           S0={fwd_swap_rate:.6f}, K={strike:.6f}")

fwd_rate base:     0.053002
fwd_rate 10y up:   0.052713
swaption base:     0.11596215
swaption 10y up:   0.11521716
S0 vs K:           S0=0.053002, K=0.053002


In [94]:
# Diagnostic clé
print(f"Swaption expiry:     {swaption_expiry}  ({year_frac_act_x(settlement_date, swaption_expiry, 365):.1f}y)")
print(f"Underlying expiry:   {underlying_expiry} ({year_frac_act_x(settlement_date, underlying_expiry, 365):.1f}y)")
print()
print("Poids 10y par maturité de swap:")
for date, w in zip(df_swaps.index, weights[0]):
    yf = year_frac_act_x(settlement_date, date, 365)
    print(f"  {yf:.1f}y : weight={w:.4f}  shock={0.01*w:.6f}")

Swaption expiry:     2018-03-15  (10.1y)
Underlying expiry:   2023-03-15 (15.1y)

Poids 10y par maturité de swap:
  1.0y : weight=1.0000  shock=0.010000
  2.0y : weight=1.0000  shock=0.010000
  3.0y : weight=1.0000  shock=0.010000
  4.0y : weight=1.0000  shock=0.010000
  5.0y : weight=1.0000  shock=0.010000
  6.0y : weight=1.0000  shock=0.010000
  7.0y : weight=1.0000  shock=0.010000
  8.0y : weight=1.0000  shock=0.010000
  9.0y : weight=1.0000  shock=0.010000
  10.0y : weight=0.9984  shock=0.009984
  11.0y : weight=0.7984  shock=0.007984
  12.0y : weight=0.5984  shock=0.005984
  13.0y : weight=0.3978  shock=0.003978
  14.0y : weight=0.1967  shock=0.001967
  15.0y : weight=0.0000  shock=0.000000
  16.0y : weight=0.0000  shock=0.000000
  17.0y : weight=0.0000  shock=0.000000
  18.0y : weight=0.0000  shock=0.000000
  19.0y : weight=0.0000  shock=0.000000
  20.0y : weight=0.0000  shock=0.000000
  21.0y : weight=0.0000  shock=0.000000
  22.0y : weight=0.0000  shock=0.000000
  23.0y : weigh

In [95]:
# =============================================================================
# Q5: Portfolio Coarse-Grained 15y Bucket DV01
# =============================================================================

fwd_swap_rate_15y_up = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_15y_up,
    swaption_underlying_fixed_leg_schedule[0],
)

# Swaption price under 15y bucket shock
swaption_price_15y_up = swaption_price_calculator(
    fwd_swap_rate_15y_up,
    strike,
    today,
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors_15y_up,
    swaption_type,
)

# IRS MtM under 15y bucket shock
irs_mtm_15y_up = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_15y_up,
)

ptf_mtm_15y_up = (
    swaption_notional * swaption_price_15y_up
    + irs_notional * irs_mtm_15y_up
)

ptf_numeric_15y_dv01 = ptf_mtm_15y_up - ptf_mtm
print(f"Portfolio Coarse-Grained 15y Bucket DV01: \u20ac{ptf_numeric_15y_dv01:,.2f}")

Portfolio Coarse-Grained 15y Bucket DV01: €558,329.13


## Q6: Delta-hedge with two liquid IRS (10y and 15y)


In [96]:
# =============================================================================
# Q6: Delta hedging with two IRS (10y and 15y)
# =============================================================================


# Idea is to cover from the shock in 15 years with a 15Y IRS, which will be sensible also to 10Y shock, 
# and then to cover the residual 10Y shock with a 10Y IRS.
b = np.array([-ptf_numeric_10y_dv01, -ptf_numeric_15y_dv01])
irs_15y_fixed_leg_payment_dates = date_series(settlement, business_date_offset(settlement, year_offset=15), irs_fixed_leg_freq)[1:]
dv01_irs_10y_up10 = irs_mtm_10y_up - irs_mtm
dv01_irs_10y_up15 = irs_mtm_15y_up - irs_mtm
irs15y_mtm = swap_mtm(irs_rate, irs_15y_fixed_leg_payment_dates, discount_factors, swap_type=SwapType.PAYER)
dv01_irs_15y_up10 = swap_mtm(irs_rate, irs_15y_fixed_leg_payment_dates, discount_factors_10y_up, swap_type=SwapType.PAYER) - irs15y_mtm
dv01_irs_15y_up15 = swap_mtm(irs_rate, irs_15y_fixed_leg_payment_dates, discount_factors_15y_up, swap_type=SwapType.PAYER) - irs15y_mtm

A = np.array([
    [dv01_irs_10y_up10, dv01_irs_15y_up10], 
    [dv01_irs_10y_up15, dv01_irs_15y_up15]  
]) # Sensibility of a 10Y IRS to a course grained DV01 in 15Y should be null


n = np.linalg.solve(A, b)
display(n)



array([-1.14903154e+09, -5.19133069e+08])

## Q7: Curve flattening scenario


In [97]:
# =============================================================================
# Q7: Curve flattening scenario
# =============================================================================
# =============================================================================
# Q7a: P&L under curve flattening - hedge with 10y IRS only
# =============================================================================

# Flattening scenario: +1bp sul bucket 10y, -1bp sul bucket 15y
# with the weights Q5
shock_flatten = pd.Series(
    +0.0001 * linear_weights(df_swaps.index, settlement_date, 10, bucket_width_y=5)
    - 0.0001 * linear_weights(df_swaps.index, settlement_date, 15, bucket_width_y=5),
    index=df_swaps.index
)

df_swaps_flatten = df_swaps.copy()
df_swaps_flatten['BID'] += shock_flatten
df_swaps_flatten['ASK'] += shock_flatten

discount_factors_flatten, _ = bootstrap(settlement_date, df_depos, df_futures, df_swaps_flatten)

# Recompute swaption price under flattened curve
fwd_swap_rate_flatten = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_flatten,
    swaption_underlying_fixed_leg_schedule[0],
)

swaption_price_flatten = swaption_price_calculator(
    fwd_swap_rate_flatten,
    strike,
    today,
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors_flatten,
    swaption_type,
)

# Recompute IRS MtM under flattened curve
irs_mtm_flatten = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_flatten
)
#correzione AI non sono troppo convinto di moltiplicare delta del punto 4 * hedge_irs_mtm_flatten
hedge_irs_mtm_flatten = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_flatten, swap_type=SwapType.RECEIVER
)

ptf_mtm_flatten_a = (
    swaption_notional * swaption_price_flatten
    + irs_notional * irs_mtm_flatten
    + delta_hedge_swap_notional * hedge_irs_mtm_flatten
)
pnl_a = ptf_mtm_flatten_a - ptf_mtm
print(f"P&L hedged portfolio (10y IRS): €{pnl_a:,.2f}")

# =============================================================================
# Q7b: P&L under curve flattening - hedge with two IRS
# =============================================================================

#ptf_mtm_flatten_b = swaption_notional * swaption_price_flatten + irs_notional * irs_mtm_flatten # should be + the two notional of point 6

# =============================================================================
# Q7b: P&L under curve flattening - hedge with two IRS (Q6)
# =============================================================================

# Nozionali dall'output di Q6
n_hedge_10y = n[0]  # notional del 10y IRS hedge
n_hedge_15y = n[1]  # notional del 15y IRS hedge

# Schedule del 15y IRS hedge (già definito in Q6)
# irs_fixed_leg_payment_dates -> 10y IRS
# irs_15y_fixed_leg_payment_dates -> 15y IRS

# Rate par per i due IRS di hedge (calcolati sulla curva base)
hedge_irs_10y_rate = swap_par_rate(irs_fixed_leg_payment_dates, discount_factors)
hedge_irs_15y_rate = swap_par_rate(irs_15y_fixed_leg_payment_dates, discount_factors)

# MtM dei due IRS di hedge sulla curva flattened
hedge_irs_10y_mtm_flatten = swap_mtm(
    hedge_irs_10y_rate,
    irs_fixed_leg_payment_dates,
    discount_factors_flatten,
    swap_type=SwapType.RECEIVER,
)
hedge_irs_15y_mtm_flatten = swap_mtm(
    hedge_irs_15y_rate,
    irs_15y_fixed_leg_payment_dates,
    discount_factors_flatten,
    swap_type=SwapType.RECEIVER,
)

# Portfolio MtM under flattening scenario hedged with two IRS
ptf_mtm_flatten_b = (
    swaption_notional * swaption_price_flatten
    + irs_notional * irs_mtm_flatten
    + n_hedge_10y * hedge_irs_10y_mtm_flatten
    + n_hedge_15y * hedge_irs_15y_mtm_flatten
)

pnl_b = ptf_mtm_flatten_b - ptf_mtm
print(f"P&L hedged portfolio (two IRS): €{pnl_b:,.2f}")

P&L hedged portfolio (10y IRS): €-2,366.30
P&L hedged portfolio (two IRS): €-2,435.55
